# 14 — Local 90/10 Ensemble Integration

This notebook consumes the successful outputs of Phase 2C (DINOv2 five-fold) and Phase 2E (ViT-B/16 full-study). It **does not rerun either model**. It validates study identity, target order, probability arrays and model status, then creates the final one-study ensemble outputs.

The blend follows the final Kaggle policy: 90% DINOv2 + 10% ViT when both are valid; use the only valid model if one branch is unavailable; use 0.5 only when neither branch is usable. A missing or malformed artifact is an error, not a silent fallback.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src" / "rsna_knee").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find project root containing src/rsna_knee. "
        "Open this notebook from the project workspace or adjust PROJECT_ROOT manually."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rsna_knee.inference.ensemble import (
    TARGETS, DEFAULT_DINO_WEIGHT, run_local_ensemble,
)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Targets:", len(TARGETS))

Project root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Python: 3.13.9
Targets: 12


## 2. Locate Phase 2C and Phase 2E outputs

These are the saved inference artifacts, not the one-slice smoke-test output. The DINOv2 input must be the **five-fold full-study prediction**, and the ViT input must be the **full-study prediction**.

In [2]:
DINOV2_OUTPUT_DIR = PROJECT_ROOT / "demo" / "output" / "dinov2_full_inference"
VIT_OUTPUT_DIR = PROJECT_ROOT / "demo" / "output" / "vit_full_inference"
OUTPUT_DIR = PROJECT_ROOT / "demo" / "output" / "ensemble"
DINOV2_WEIGHT = 0.90

required_inputs = {
    "DINOv2 five-fold NPZ": DINOV2_OUTPUT_DIR / "dinov2_five_fold_predictions.npz",
    "DINOv2 report": DINOV2_OUTPUT_DIR / "dinov2_five_fold_inference_report.json",
    "ViT full-study NPZ": VIT_OUTPUT_DIR / "vit_full_study_prediction.npz",
    "ViT report": VIT_OUTPUT_DIR / "vit_full_study_inference_report.json",
}
for label, path in required_inputs.items():
    print(f"{'[FOUND]' if path.is_file() else '[MISSING]'} {label}: {path}")
assert all(path.is_file() for path in required_inputs.values()), (
    "Missing Phase 2C/2E outputs. Run notebooks 10 and 13 successfully first."
)
assert 0.0 <= DINOV2_WEIGHT <= 1.0

[FOUND] DINOv2 five-fold NPZ: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_full_inference\dinov2_five_fold_predictions.npz
[FOUND] DINOv2 report: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\dinov2_full_inference\dinov2_five_fold_inference_report.json
[FOUND] ViT full-study NPZ: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\vit_full_inference\vit_full_study_prediction.npz
[FOUND] ViT report: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\vit_full_inference\vit_full_study_inference_report.json


## 3. Validate and blend model predictions

The integration module checks:
- the same `StudyInstanceUID` in both outputs;
- the exact canonical order of all 12 targets;
- DINOv2's five fold probabilities and their saved arithmetic mean;
- strict-load status for all five DINOv2 folds and the final ViT checkpoint;
- finite probabilities in `[0, 1]`;
- the per-study fallback policy.

In [3]:
prediction, output_paths = run_local_ensemble(
    dinov2_output_dir=DINOV2_OUTPUT_DIR,
    vit_output_dir=VIT_OUTPUT_DIR,
    output_dir=OUTPUT_DIR,
    dinov2_weight=DINOV2_WEIGHT,
)

print("========== LOCAL ENSEMBLE ==========")
print("Study UID:", prediction.study_uid)
print("DINOv2 valid:", prediction.dinov2_valid, "| status:", prediction.dino_status)
print("ViT valid:", prediction.vit_valid, "| status:", prediction.vit_status)
print("Blend status:", prediction.blend_status)
print("Configured weights:", prediction.configured_dinov2_weight, prediction.configured_vit_weight)
print("Effective weights:", prediction.effective_dinov2_weight, prediction.effective_vit_weight)
print("Output shape:", prediction.probabilities.shape)
print("Probability range:", float(prediction.probabilities.min()), "to", float(prediction.probabilities.max()))

result_table = pd.DataFrame({
    "Target": list(prediction.targets),
    "DINOv2 probability": prediction.dinov2_probabilities,
    "ViT-B/16 probability": prediction.vit_probabilities,
    "Ensemble probability": prediction.probabilities,
})
display(result_table)

========== LOCAL ENSEMBLE ==========
Study UID: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
DINOv2 valid: True | status: ok
ViT valid: True | status: success
Blend status: weighted_ensemble
Configured weights: 0.9 0.09999999999999998
Effective weights: 0.9 0.09999999999999998
Output shape: (12,)
Probability range: 0.20235773921012878 to 0.5219954252243042


,Target,DINOv2 probability,ViT-B/16 probability,Ensemble probability
0,ACL,0.236336,0.327129,0.245416
1,MCL,0.290734,0.209639,0.282625
2,Medial Meniscus,0.249113,0.314138,0.255615
3,Lateral Meniscus,0.263609,0.335217,0.270770
4,Medial OA,0.372019,0.341920,0.369009
5,Lateral OA,0.457481,0.338576,0.445590
6,PF OA,0.534415,0.410223,0.521995
7,Effusion,0.527459,0.313934,0.506106
8,Synovitis,0.320996,0.279719,0.316868
9,Baker's,0.292300,0.355100,0.298580


## 4. Verify the blend formula and saved outputs

In [4]:
if prediction.dinov2_valid and prediction.vit_valid:
    expected = (
        prediction.configured_dinov2_weight * prediction.dinov2_probabilities
        + prediction.configured_vit_weight * prediction.vit_probabilities
    )
elif prediction.dinov2_valid:
    expected = prediction.dinov2_probabilities
elif prediction.vit_valid:
    expected = prediction.vit_probabilities
else:
    expected = np.full(len(TARGETS), 0.5, dtype=np.float32)

assert np.allclose(prediction.probabilities, expected, rtol=1e-6, atol=1e-7)
assert prediction.probabilities.shape == (12,)
assert np.isfinite(prediction.probabilities).all()
assert np.all((prediction.probabilities >= 0) & (prediction.probabilities <= 1))

print("[PASS] Effective fallback/weighted formula verified")
print("Saved files:")
for name, path in output_paths.items():
    print(f"  {name}: {path}")
assert all(path.is_file() for path in output_paths.values())

[PASS] Effective fallback/weighted formula verified
Saved files:
  wide_csv: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\ensemble\ensemble_prediction.csv
  long_csv: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\ensemble\ensemble_prediction_long.csv
  npz: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\ensemble\ensemble_prediction.npz
  report: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\ensemble\ensemble_report.json


## 5. Inspect final competition-shaped CSV and detailed report

In [5]:
wide_df = pd.read_csv(output_paths["wide_csv"])
long_df = pd.read_csv(output_paths["long_csv"])
report = json.loads(output_paths["report"].read_text(encoding="utf-8"))

print("Competition-shaped CSV:")
display(wide_df)
print("Detailed target-level CSV:")
display(long_df)
print("Report status:", report["status"])
print("Blend status:", report["blend_status"])
print("Output summary:", report["output"])

Competition-shaped CSV:


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10095687747295410396...,0.245416,0.282625,0.255615,0.27077,0.369009,0.44559,0.521995,0.506106,0.316868,0.29858,0.324787,0.202358


Detailed target-level CSV:


,StudyInstanceUID,target,dinov2_probability,vit_probability,dinov2_valid,vit_valid,effective_dinov2_weight,effective_vit_weight,ensemble_probability,blend_status
0,1.2.826.0.1.3680043.8.498.10095687747295410396...,ACL,0.236336,0.327129,True,True,0.9,0.1,0.245416,weighted_ensemble
1,1.2.826.0.1.3680043.8.498.10095687747295410396...,MCL,0.290734,0.209639,True,True,0.9,0.1,0.282625,weighted_ensemble
2,1.2.826.0.1.3680043.8.498.10095687747295410396...,Medial Meniscus,0.249113,0.314138,True,True,0.9,0.1,0.255615,weighted_ensemble
3,1.2.826.0.1.3680043.8.498.10095687747295410396...,Lateral Meniscus,0.263609,0.335217,True,True,0.9,0.1,0.270770,weighted_ensemble
4,1.2.826.0.1.3680043.8.498.10095687747295410396...,Medial OA,0.372019,0.341920,True,True,0.9,0.1,0.369009,weighted_ensemble
5,1.2.826.0.1.3680043.8.498.10095687747295410396...,Lateral OA,0.457481,0.338576,True,True,0.9,0.1,0.445590,weighted_ensemble
6,1.2.826.0.1.3680043.8.498.10095687747295410396...,PF OA,0.534415,0.410223,True,True,0.9,0.1,0.521995,weighted_ensemble
7,1.2.826.0.1.3680043.8.498.10095687747295410396...,Effusion,0.527459,0.313934,True,True,0.9,0.1,0.506106,weighted_ensemble
8,1.2.826.0.1.3680043.8.498.10095687747295410396...,Synovitis,0.320996,0.279719,True,True,0.9,0.1,0.316868,weighted_ensemble
9,1.2.826.0.1.3680043.8.498.10095687747295410396...,Baker's,0.292300,0.355100,True,True,0.9,0.1,0.298580,weighted_ensemble


Report status: success
Blend status: weighted_ensemble
Output summary: {'shape': [12], 'all_finite': True, 'probability_min': 0.20235773921012878, 'probability_max': 0.5219954252243042}


## 6. CRITERIA — Phase 3 ensemble integration

All criteria must pass before moving to the demo UI.

In [6]:
report = json.loads(output_paths["report"].read_text(encoding="utf-8"))
with np.load(output_paths["npz"], allow_pickle=False) as saved:
    saved_targets = saved["targets"].astype(str).tolist()
    saved_probabilities = saved["ensemble_probabilities"].copy()
    saved_uid = str(saved["study_uid"].item())
    saved_weights = saved["effective_weights"].copy()

criteria = {
    "Both model artifacts loaded and validated": True,
    "DINOv2 and ViT StudyInstanceUID match": prediction.study_uid == saved_uid,
    "Canonical 12-target order preserved": saved_targets == list(TARGETS),
    "DINOv2 five-fold artifact and mean contract validated": (
        prediction.dino_status in {"ok", "all_roles_unusable_fallback_0.5"}
    ),
    "DINOv2/ViT validity and fallback policy applied": (
        prediction.blend_status in {"weighted_ensemble", "dinov2_only", "vit_only", "neutral_fallback_0.5"}
    ),
    "Effective weights match blend status": (
        np.allclose(saved_weights, [prediction.effective_dinov2_weight, prediction.effective_vit_weight])
    ),
    "Final prediction has 12 finite probabilities in [0, 1]": (
        saved_probabilities.shape == (12,)
        and np.isfinite(saved_probabilities).all()
        and np.all((saved_probabilities >= 0) & (saved_probabilities <= 1))
    ),
    "Weighted/fallback formula verified": np.allclose(saved_probabilities, expected, rtol=1e-6, atol=1e-7),
    "All four output files saved": all(path.is_file() for path in output_paths.values()),
    "Report records correct status": report["status"] == ("success" if prediction.dinov2_valid or prediction.vit_valid else "fallback"),
}

for name, passed in criteria.items():
    print(f"{'[PASS]' if passed else '[FAIL]'} {name}")

assert all(criteria.values()), "Phase 3 did not pass every completion criterion."
print("\nPHASE 3: ALL CRITERIA PASSED")

[PASS] Both model artifacts loaded and validated
[PASS] DINOv2 and ViT StudyInstanceUID match
[PASS] Canonical 12-target order preserved
[PASS] DINOv2 five-fold artifact and mean contract validated
[PASS] DINOv2/ViT validity and fallback policy applied
[PASS] Effective weights match blend status
[PASS] Final prediction has 12 finite probabilities in [0, 1]
[PASS] Weighted/fallback formula verified
[PASS] All four output files saved
[PASS] Report records correct status

PHASE 3: ALL CRITERIA PASSED


## Important interpretation note

This output is a one-study software demonstration using the selected `gold_validation` sample. It is not a new competition score, an independent validation estimate, or a clinical diagnosis. Keep the DINOv2 and ViT source outputs unchanged for traceability.